# 固定样本的数据质量审计

## tl;dr
固定样本包含90,750条评论、1,000个ASIN。五字段重复副本231条，同商品文本重复额外632条；全样本文本重复6,855条不能替代同商品口径。600对AI辅助标签已经完成；回看后验证确定正例11对，未达到20对门槛，模型训练与最终测试暂停。

## Context & Methods
按ASIN划分70/15/15，跨集合相同规范化正文进入公共模板池；五字段副本不进入监督记录池。研究内容冗余，不判断作者身份。

### Key Assumptions
有界流式扫描、局部缓冲区打乱、优先选择观测频次较高ASIN并每商品最多100条。不是全类别均匀样本。原下载提交未记录，parent_asin未保留。标签由助手及独立子任务盲审，未经独立人类金标准复核。时间统一显示UTC，原始字段保持整数毫秒。

In [ ]:
from pathlib import Path
import sys
import pandas as pd
from IPython.display import display
root = next(p for p in [Path.cwd(), Path.cwd().parent] if (p / "src/common.py").exists())
sys.path.insert(0, str(root))
from src.common import load_study, read_json, COLUMNS
df, manifest = load_study(root)
print("Dataset SHA-256:", manifest["dataset_version"])

## Data
### 1. 核对字段、基本规模和时间范围

In [ ]:
display(pd.DataFrame({"field": COLUMNS, "dtype": [str(df[c].dtype) for c in COLUMNS], "empty_rows": [int(df[c].astype(str).eq("").sum()) for c in COLUMNS]}))
display(pd.DataFrame([{"reviews": len(df), "ASINs": df.asin.nunique(), "users": df.user_id.nunique(), "earliest_UTC": pd.to_datetime(df.timestamp.min(), unit="ms", utc=True), "latest_UTC": pd.to_datetime(df.timestamp.max(), unit="ms", utc=True)}]))
assert df.review_id.is_unique
assert df.rating.isin([1,2,3,4,5]).all()
assert df.timestamp.mod(1).eq(0).all()

## Results
### 2. 重算三个重复口径

In [ ]:
counts = {"five_field_extra_rows": int(df.source_id.duplicated().sum()), "within_ASIN_text_extra_rows": int(df.duplicated(["asin", "text_hash"]).sum()), "global_text_extra_rows": int(df.text_hash.duplicated().sum())}
display(pd.DataFrame([{"definition": k, "extra_rows": v, "share_of_all_rows": v/len(df)} for k,v in counts.items()]))
assert list(counts.values()) == [231, 632, 6855]

### 3. 检查分组与模板隔离

In [ ]:
eligible = df[df.supervised_eligible]
for field in ["asin", "text_hash", "source_id"]:
    assert eligible.groupby(field).split.nunique().max() <= 1
display(df.groupby("split").agg(rows=("review_id","size"), ASINs=("asin","nunique"), supervised_rows=("supervised_eligible","sum")).reset_index())

### 4. 清洗差异、截断与标注门槛
下表仅读取训练和验证标签的汇总；测试标签保持封存。

In [ ]:
display(pd.DataFrame([read_json(root / "results/tables/cleaning_audit.json")]).drop(columns=["reason_counts_overlapping"]))
display(pd.read_csv(root / "results/tables/annotation_label_counts.csv"))
semantic = read_json(root / "models/embedding_manifest.json")
print("Truncated reviews:", semantic["truncated_rows"], "/", semantic["rows"])
print("Pinned encoder revision:", semantic["model_revision"])

### 5. 读取实际验收结果和性能记录

In [ ]:
display(pd.DataFrame(read_json(root / "results/tables/acceptance_report.json")["checks"]))
display(pd.read_csv(root / "results/tables/cached_matrix_benchmark.csv")[["products","reviews","pairs","seconds","peak_rss_mb","scope"]])

## Takeaways
数据版本、唯一编号、按商品配对及文本泄漏检查通过。当前瓶颈是固定验证集的确定正例不足；不得通过猜标签或放宽定义训练。改进清洗保留尖括号及比较符号，12条正文与旧版本不同，原CSV保持不变并提供差异表。MiniLM超过256word pieces会截断，本样本1,324条受影响；标签集有困难样本富集，标签重复比例不是总体自然重复率。

参考：[Amazon数据卡](https://huggingface.co/datasets/McAuley-Lab/Amazon-Reviews-2023)、[MiniLM模型说明](https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2)、[scikit-learn分组划分原则](https://scikit-learn.org/stable/modules/cross_validation.html)。